# AutoShield AI: Automotive Supply Chain Digital Twin

## Project Overview

AutoShield AI is an AI-powered supply chain risk intelligence platform designed for the automotive industry.

Modern automotive manufacturing depends on globally distributed suppliers for semiconductors, battery materials, mechanical assemblies, and other critical components. Disruptions in these supply networks can lead to production delays, increased operational costs, and revenue loss.

This notebook focuses on constructing a digital representation of the supplier ecosystem using historical supply chain transaction data.

The generated digital twin serves as the analytical foundation for:

- Supplier Risk Monitoring
- Alternative Sourcing Recommendations
- Scenario Simulation
- Executive Decision Support

---

## Objectives

The primary objectives of this notebook are:

1. Load and explore supply chain transaction data.
2. Clean and prepare the dataset.
3. Engineer business-relevant supply chain features.
4. Create automotive supplier and commodity mappings.
5. Build a supplier intelligence layer for downstream analytics.
6. Export a structured dataset for risk modeling and decision support systems.

---

## Expected Outputs

This notebook produces:

- Cleaned Supply Chain Dataset
- Supplier Mapping Layer
- Commodity Classification Layer
- Delivery Delay Metrics
- Automotive Supply Chain Digital Twin Dataset

# 1. Import Required Libraries

This section imports the core libraries used throughout the project.

### Libraries

- pandas: Data manipulation and analysis
- numpy: Numerical computations
- matplotlib: Data visualization
- seaborn: Statistical visualizations

These libraries provide the foundation for data exploration, preprocessing, and feature engineering.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# 2. Load Supply Chain Dataset

The dataset contains historical supply chain transactions, including:

- Shipping Information
- Delivery Performance
- Customer Geography
- Product Information
- Order Details

This data will be transformed into an automotive supplier intelligence dataset.

In [2]:
df = pd.read_csv(
    "../data/raw/DataCoSupplyChainDataset.csv",
    encoding="latin1"
)

# 3. Initial Dataset Exploration

Before feature engineering, it is important to understand:

- Dataset size
- Column availability
- Data types
- Missing values
- Business relevance of attributes

This step helps identify potential data quality issues and guides subsequent preprocessing decisions.

In [3]:
print(df.shape)

df.head()

(180519, 53)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  object 
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  object 
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  object 
 9   Customer City                  180519 non-null  object 
 10  Customer Country               180519 non-null  object 
 11  Customer Email                 180519 non-null  object 
 12  Customer Fname                

In [5]:
df.isnull().sum()

Type                                  0
Days for shipping (real)              0
Days for shipment (scheduled)         0
Benefit per order                     0
Sales per customer                    0
Delivery Status                       0
Late_delivery_risk                    0
Category Id                           0
Category Name                         0
Customer City                         0
Customer Country                      0
Customer Email                        0
Customer Fname                        0
Customer Id                           0
Customer Lname                        8
Customer Password                     0
Customer Segment                      0
Customer State                        0
Customer Street                       0
Customer Zipcode                      3
Department Id                         0
Department Name                       0
Latitude                              0
Longitude                             0
Market                                0


# Key Observation

The dataset contains over 180,000 supply chain transactions covering multiple regions, countries, product categories, and delivery outcomes.

Several attributes contain customer-specific information that is not directly relevant for supply chain risk analytics and may be excluded from downstream modeling.

# 4. Delivery Performance Engineering

Delivery performance is one of the strongest indicators of supplier reliability.

To quantify delivery efficiency, a new metric called **Delay_Days** is created.

## Delay_Days

Delay_Days = Actual Shipping Time - Scheduled Shipping Time

Interpretation:

- Negative Value → Delivered Earlier Than Expected
- Zero → Delivered On Time
- Positive Value → Delivered Late

This feature serves as a core signal for disruption risk analysis.

In [6]:
df["Delay_Days"] = (
    df["Days for shipping (real)"]
    -
    df["Days for shipment (scheduled)"]
)

In [7]:
df["Delay_Days"].describe()

count    180519.000000
mean          0.565807
std           1.490966
min          -2.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           4.000000
Name: Delay_Days, dtype: float64

# Business Interpretation

Higher positive values indicate operational inefficiencies and delivery disruptions.

These disruptions may originate from:

- Supplier constraints
- Logistics bottlenecks
- Inventory shortages
- Transportation delays

As a result, Delay_Days is expected to be an important predictor of supply chain risk.

# 5. Automotive Commodity Mapping

The original dataset contains retail-oriented product categories.

To align the dataset with the automotive domain, product categories are mapped to automotive component groups.

## Automotive Commodity Categories

- Semiconductors
- Battery Materials
- Mechanical Assemblies
- Metal Components
- Plastic Components
- Interior Components

These categories represent major sourcing areas within modern automotive supply chains.

In [8]:
automotive_categories = [
    "Semiconductors",
    "Battery Materials",
    "Mechanical Assemblies",
    "Metal Components",
    "Plastic Components",
    "Interior Components"
]

In [9]:
df["Commodity"] = np.select(
    [
        df["Product Category Id"] % 6 == 0,
        df["Product Category Id"] % 6 == 1,
        df["Product Category Id"] % 6 == 2,
        df["Product Category Id"] % 6 == 3,
        df["Product Category Id"] % 6 == 4,
        df["Product Category Id"] % 6 == 5,
    ], automotive_categories
)

In [10]:
df["Commodity"].value_counts()

Commodity
Semiconductors           62513
Interior Components      39501
Metal Components         32820
Plastic Components       22685
Battery Materials        18791
Mechanical Assemblies     4209
Name: count, dtype: int64

# 6. Supplier Network Construction

A supplier network is created using country-level sourcing relationships.

Each supplier represents a sourcing location within the global automotive supply chain.

This abstraction enables:

- Geographic Risk Analysis
- Supplier Diversification Studies
- Alternative Sourcing Recommendations
- Scenario-Based Simulations

In [11]:
df["Supplier"] = df["Order Country"]

In [12]:
df["Supplier"].value_counts().head(20)

Supplier
Estados Unidos          24840
Francia                 13222
México                  13172
Alemania                 9564
Australia                8497
Brasil                   7987
Reino Unido              7302
China                    5758
Italia                   4989
India                    4783
Indonesia                4204
España                   3868
El Salvador              3726
República Dominicana     3669
Honduras                 3629
Cuba                     3534
Turquía                  3395
Nicaragua                3046
Guatemala                2778
Nigeria                  2309
Name: count, dtype: int64

# 7. Supply Chain Digital Twin Creation

The digital twin combines:

- Supplier Information
- Commodity Categories
- Delivery Performance
- Order Volume
- Revenue Contribution

This creates a structured representation of the automotive supply ecosystem that can be used for advanced analytics and AI-driven decision support.

In [13]:
digital_twin = df[
    [
        "Order Id",
        "Supplier",
        "Commodity",
        "Order Country",
        "Order Region",
        "Shipping Mode",
        "Days for shipping (real)",
        "Days for shipment (scheduled)",
        "Delay_Days",
        "Late_delivery_risk",
        "Order Item Quantity",
        "Sales"
    ]
].copy()

In [14]:
print(digital_twin.shape)

digital_twin.head(15)

(180519, 12)


,Order Id,Supplier,Commodity,Order Country,Order Region,Shipping Mode,Days for shipping (real),Days for shipment (scheduled),Delay_Days,Late_delivery_risk,Order Item Quantity,Sales
0,77202,Indonesia,Battery Materials,Indonesia,Southeast Asia,Standard Class,3,4,-1,0,1,327.75
1,75939,India,Battery Materials,India,South Asia,Standard Class,5,4,1,1,1,327.75
2,75938,India,Battery Materials,India,South Asia,Standard Class,4,4,0,0,1,327.75
3,75937,Australia,Battery Materials,Australia,Oceania,Standard Class,3,4,-1,0,1,327.75
4,75936,Australia,Battery Materials,Australia,Oceania,Standard Class,2,4,-2,0,1,327.75
5,75935,Australia,Battery Materials,Australia,Oceania,Standard Class,6,4,2,0,1,327.75
6,75934,China,Battery Materials,China,Eastern Asia,First Class,2,1,1,1,1,327.75
7,75933,China,Battery Materials,China,Eastern Asia,First Class,2,1,1,1,1,327.75
8,75932,China,Battery Materials,China,Eastern Asia,Second Class,3,2,1,1,1,327.75
9,75931,China,Battery Materials,China,Eastern Asia,First Class,2,1,1,1,1,327.75


In [15]:
supplier_summary = (
    digital_twin.groupby("Supplier")
    .agg(
        orders=("Order Id","count"),
        avg_delay=("Delay_Days","mean"),
        late_rate=("Late_delivery_risk","mean"),
        sales=("Sales","sum")
    )
    .reset_index()
)

supplier_summary.head()

,Supplier,orders,avg_delay,late_rate,sales
0,Afganistán,163,0.607362,0.570552,3.856886e+04
1,Albania,37,0.351351,0.594595,8.299130e+03
2,Alemania,9564,0.593266,0.562840,2.074172e+06
3,Angola,306,0.879085,0.598039,5.766602e+04
4,Arabia Saudí,860,0.559302,0.576744,1.631497e+05


# 8. Export Processed Dataset

The final digital twin dataset is exported for use in subsequent modules:

- Supplier Risk Prediction Agent
- Alternative Sourcing Engine
- Scenario Simulation Engine
- Executive AI Copilot

This exported dataset becomes the foundation of the AutoShield AI platform.

In [16]:
digital_twin.to_csv(
    "../data/processed/automotive_digital_twin.csv",
    index=False
)

# Conclusion

In this notebook, a raw supply chain transaction dataset was transformed into an automotive supply chain digital twin.

Key accomplishments include:

- Delivery performance engineering
- Automotive commodity classification
- Supplier network construction
- Digital twin generation

The resulting dataset provides the foundation for risk prediction, disruption simulation, and supply chain decision intelligence within the AutoShield AI platform.